In [ ]:
!pip install catboost -q

In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import warnings
from tqdm.auto import tqdm
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score
from sklearn.utils import resample
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')


In [ ]:
def inject_missing_data(X, ratio, random_seed=42):
    if ratio == 0:
        return X.copy()
    np.random.seed(random_seed)
    X_miss = X.copy()
    mask = np.random.rand(*X_miss.shape) < ratio
    X_miss = X_miss.mask(mask)
    return X_miss

def evaluate_with_bootstrap(model, X_test, y_test, n_iterations=1000):
    metrics = {'roc_auc': [], 'f1': [], 'accuracy': [], 'precision': [], 'recall': []}
    is_multiclass = len(np.unique(y_test)) > 2
    avg_method = 'macro' if is_multiclass else 'binary'

    for _ in range(n_iterations):
        X_boot, y_boot = resample(X_test, y_test)
        if len(np.unique(y_boot)) < 2: continue

        y_pred = model.predict(X_boot)
        y_prob = model.predict_proba(X_boot)

        try:
            if is_multiclass: metrics['roc_auc'].append(roc_auc_score(y_boot, y_prob, multi_class='ovr'))
            else: metrics['roc_auc'].append(roc_auc_score(y_boot, y_prob[:, 1]))

            metrics['f1'].append(f1_score(y_boot, y_pred, average=avg_method, zero_division=0))
            metrics['accuracy'].append(accuracy_score(y_boot, y_pred))
            metrics['precision'].append(precision_score(y_boot, y_pred, average=avg_method, zero_division=0))
            metrics['recall'].append(recall_score(y_boot, y_pred, average=avg_method, zero_division=0))
        except: continue

    results = {}
    for k, v in metrics.items():
        if len(v) == 0: results[k] = "N/A"
        else: results[k] = f"{np.mean(v):.4f}±{1.96 * np.std(v):.4f}"
    return results

def load_and_prep_data(dataset_name):
    dataset_ids = {'Bank': 1461, 'Blood': 1464, 'California': 44090, 'Car': 40975,
                   'Credit-G': 31, 'Diabetes': 37, 'Heart': 53, 'Income': 1590, 'Jungle': 41027}
    if dataset_name == 'Diabetes':
        path = kagglehub.dataset_download("uciml/pima-indians-diabetes-database") + "/diabetes.csv"
        df = pd.read_csv(path); X, y = df.drop(columns=['Outcome']), df['Outcome']
    elif dataset_name == 'Heart':
        path = kagglehub.dataset_download("fedesoriano/heart-failure-prediction") + "/heart.csv"
        df = pd.read_csv(path); X, y = df.drop(columns=['HeartDisease']), df['HeartDisease']
    else:
        data = fetch_openml(data_id=dataset_ids[dataset_name], as_frame=True, parser='auto')
        X, y = data.data, data.target
    le = LabelEncoder()
    y = le.fit_transform(y.astype(str))
    return X, y

def run_logistic_regression(X_train, X_test, y_train, y_test):
    cat_cols = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    num_cols = X_train.select_dtypes(exclude=['object', 'category', 'string']).columns.tolist()
    num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
                         ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    preprocessor = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)])
    pipe = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
    pipe.fit(X_train, y_train)
    return evaluate_with_bootstrap(pipe, X_test, y_test)

def run_catboost(X_train, X_test, y_train, y_test):
    cat_features = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    X_tr, X_te = X_train.copy(), X_test.copy()
    for col in cat_features:
        X_tr[col] = X_tr[col].astype(str).replace('nan', 'missing')
        X_te[col] = X_te[col].astype(str).replace('nan', 'missing')
    model = CatBoostClassifier(iterations=1000, learning_rate=0.1, depth=6, cat_features=cat_features,
                               verbose=0, random_seed=42, nan_mode='Min', task_type='GPU')
    model.fit(X_tr, y_train)
    return evaluate_with_bootstrap(model, X_te, y_test)

def run_lightgbm(X_train, X_test, y_train, y_test):
    X_tr, X_te = X_train.copy(), X_test.copy()
    cat_cols = X_tr.select_dtypes(include=['object', 'string', 'category']).columns
    for col in cat_cols:
        X_tr[col] = X_tr[col].astype('category')
        X_te[col] = X_te[col].astype('category')
    model = LGBMClassifier(n_estimators=1000, random_state=42, verbose=-1, device='gpu')
    model.fit(X_tr, y_train)
    return evaluate_with_bootstrap(model, X_te, y_test)

In [ ]:
datasets = ["Bank", "Blood", "California", "Car", "Credit-G", "Diabetes", "Heart", "Income", "Jungle"]
missing_ratios = [0.0, 0.2, 0.5, 0.9]

preloaded_data = {}
for ds_name in tqdm(datasets, desc="Подготовка данных"):
    preloaded_data[ds_name] = load_and_prep_data(ds_name)

Подготовка данных:   0%|          | 0/9 [00:00<?, ?it/s]

Using Colab cache for faster access to the 'pima-indians-diabetes-database' dataset.
Using Colab cache for faster access to the 'heart-failure-prediction' dataset.


In [ ]:
# Logistic Regression
lr_results = []
for ds_name in tqdm(datasets, desc="Logistic Regression"):
    X_orig, y = preloaded_data[ds_name]
    X_train_val, X_test_clean, y_train_val, y_test = train_test_split(X_orig, y, test_size=0.2, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val)

    for ratio in tqdm(missing_ratios):
        X_test_noisy = inject_missing_data(X_test_clean, ratio, random_seed=42)
        res = run_logistic_regression(X_train, X_test_noisy, y_train, y_test)
        lr_results.append({'Dataset': ds_name, 'Missing_Ratio': f"{int(ratio*100)}%", 'Model': 'LogReg', **res})

lr_df = pd.DataFrame(lr_results)

display(lr_df)

Logistic Regression:   0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

...

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

,Dataset,Missing_Ratio,Model,roc_auc,f1,accuracy,precision,recall
0,Bank,0%,LogReg,0.9057±0.0092,0.4496±0.0306,0.9014±0.0060,0.6480±0.0402,0.3444±0.0290
1,Bank,20%,LogReg,0.8583±0.0122,0.4404±0.0296,0.8972±0.0060,0.6060±0.0395,0.3461±0.0282
2,Bank,50%,LogReg,0.7779±0.0159,0.3698±0.0304,0.8887±0.0063,0.5474±0.0436,0.2794±0.0268
3,Bank,90%,LogReg,0.6090±0.0182,0.1284±0.0257,0.8802±0.0066,0.4303±0.0716,0.0755±0.0161
4,Blood,0%,LogReg,0.7869±0.0889,0.1809±0.1546,0.7685±0.0658,0.5654±0.4052,0.1101±0.1019
5,Blood,20%,LogReg,0.7913±0.0753,0.2295±0.1668,0.7862±0.0656,0.8345±0.3363,0.1359±0.1109
6,Blood,50%,LogReg,0.7499±0.0794,0.0979±0.1300,0.7667±0.0668,0.6340±0.6748,0.0542±0.0754
7,Blood,90%,LogReg,0.5854±0.0914,0.0000±0.0000,0.7536±0.0683,0.0000±0.0000,0.0000±0.0000
8,California,0%,LogReg,0.9138±0.0083,0.8374±0.0119,0.8365±0.0110,0.8320±0.0163,0.8428±0.0155
9,California,20%,LogReg,0.7876±0.0136,0.7413±0.0146,0.7393±0.0133,0.7351±0.0191,0.7477±0.0186


In [ ]:
# CatBoost
cat_results = []
for ds_name in tqdm(datasets, desc="CatBoost"):
    X_orig, y = preloaded_data[ds_name]
    X_train_val, X_test_clean, y_train_val, y_test = train_test_split(X_orig, y, test_size=0.2, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val)

    for ratio in tqdm(missing_ratios):
        X_test_noisy = inject_missing_data(X_test_clean, ratio, random_seed=42)
        res = run_catboost(X_train, X_test_noisy, y_train, y_test)
        cat_results.append({'Dataset': ds_name, 'Missing_Ratio': f"{int(ratio*100)}%", 'Model': 'CatBoost', **res})

cat_df = pd.DataFrame(cat_results)

display(cat_df)

CatBoost:   0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

...

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

,Dataset,Missing_Ratio,Model,roc_auc,f1,accuracy,precision,recall
0,Bank,0%,CatBoost,0.9307±0.0066,0.5422±0.0273,0.9075±0.0059,0.6431±0.0335,0.4688±0.0298
1,Bank,20%,CatBoost,0.7625±0.0155,0.3701±0.0248,0.8468±0.0073,0.3565±0.0267,0.3850±0.0287
2,Bank,50%,CatBoost,0.6474±0.0189,0.2522±0.0270,0.8520±0.0073,0.3082±0.0341,0.2136±0.0250
3,Bank,90%,CatBoost,0.5264±0.0179,0.0150±0.0100,0.8826±0.0064,0.4024±0.2211,0.0076±0.0051
4,Blood,0%,CatBoost,0.7455±0.0972,0.4545±0.1564,0.7816±0.0680,0.5601±0.1995,0.3882±0.1598
5,Blood,20%,CatBoost,0.6168±0.1066,0.1604±0.1337,0.7272±0.0700,0.3093±0.2507,0.1108±0.0990
6,Blood,50%,CatBoost,0.5631±0.1063,0.0969±0.1192,0.7535±0.0680,0.3936±0.4648,0.0565±0.0724
7,Blood,90%,CatBoost,0.5239±0.0963,0.0000±0.0000,0.7602±0.0681,0.0000±0.0000,0.0000±0.0000
8,California,0%,CatBoost,0.9618±0.0051,0.8933±0.0098,0.8927±0.0093,0.8870±0.0135,0.8998±0.0131
9,California,20%,CatBoost,0.7970±0.0134,0.7764±0.0127,0.7472±0.0128,0.6956±0.0172,0.8786±0.0138


In [ ]:
# LightGBM
lgb_results = []
for ds_name in tqdm(datasets, desc="LightGBM"):
    X_orig, y = preloaded_data[ds_name]
    X_train_val, X_test_clean, y_train_val, y_test = train_test_split(X_orig, y, test_size=0.2, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val)

    for ratio in tqdm(missing_ratios):
        X_test_noisy = inject_missing_data(X_test_clean, ratio, random_seed=42)
        res = run_lightgbm(X_train, X_test_noisy, y_train, y_test)
        lgb_results.append({'Dataset': ds_name, 'Missing_Ratio': f"{int(ratio*100)}%", 'Model': 'LightGBM', **res})

lgb_df = pd.DataFrame(lgb_results)

display(lgb_df)

LightGBM:   0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

...

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

,Dataset,Missing_Ratio,Model,roc_auc,f1,accuracy,precision,recall
0,Bank,0%,LightGBM,0.9238±0.0070,0.5423±0.0272,0.9069±0.0058,0.6372±0.0332,0.4723±0.0303
1,Bank,20%,LightGBM,0.7904±0.0156,0.4098±0.0281,0.8802±0.0065,0.4834±0.0343,0.3559±0.0288
2,Bank,50%,LightGBM,0.6428±0.0195,0.2579±0.0287,0.8706±0.0068,0.3917±0.0415,0.1924±0.0240
3,Bank,90%,LightGBM,0.5073±0.0194,0.0347±0.0156,0.8821±0.0065,0.4064±0.1449,0.0181±0.0083
4,Blood,0%,LightGBM,0.7017±0.0964,0.4426±0.1441,0.7543±0.0697,0.4817±0.1746,0.4155±0.1554
5,Blood,20%,LightGBM,0.6585±0.1076,0.4008±0.1493,0.7260±0.0713,0.4220±0.1715,0.3878±0.1634
6,Blood,50%,LightGBM,0.5972±0.1160,0.3612±0.1464,0.6997±0.0749,0.3698±0.1619,0.3593±0.1644
7,Blood,90%,LightGBM,0.5109±0.0876,0.0911±0.1158,0.7400±0.0693,0.2868±0.3595,0.0556±0.0738
8,California,0%,LightGBM,0.9656±0.0050,0.9033±0.0094,0.9029±0.0089,0.8983±0.0132,0.9085±0.0125
9,California,20%,LightGBM,0.7547±0.0149,0.7351±0.0150,0.7426±0.0128,0.7566±0.0187,0.7150±0.0191


In [ ]:
display(lr_df.head())
display(cat_df.head())
display(lgb_df.head())

lr_df.to_csv("results_logistic_regression.csv", index=False)
cat_df.to_csv("results_catboost.csv", index=False)
lgb_df.to_csv("results_lightgbm.csv", index=False)

,Dataset,Missing_Ratio,Model,roc_auc,f1,accuracy,precision,recall
0,Bank,0%,LogReg,0.9057±0.0092,0.4496±0.0306,0.9014±0.0060,0.6480±0.0402,0.3444±0.0290
1,Bank,20%,LogReg,0.8583±0.0122,0.4404±0.0296,0.8972±0.0060,0.6060±0.0395,0.3461±0.0282
2,Bank,50%,LogReg,0.7779±0.0159,0.3698±0.0304,0.8887±0.0063,0.5474±0.0436,0.2794±0.0268
3,Bank,90%,LogReg,0.6090±0.0182,0.1284±0.0257,0.8802±0.0066,0.4303±0.0716,0.0755±0.0161
4,Blood,0%,LogReg,0.7869±0.0889,0.1809±0.1546,0.7685±0.0658,0.5654±0.4052,0.1101±0.1019


,Dataset,Missing_Ratio,Model,roc_auc,f1,accuracy,precision,recall
0,Bank,0%,CatBoost,0.9307±0.0066,0.5422±0.0273,0.9075±0.0059,0.6431±0.0335,0.4688±0.0298
1,Bank,20%,CatBoost,0.7625±0.0155,0.3701±0.0248,0.8468±0.0073,0.3565±0.0267,0.3850±0.0287
2,Bank,50%,CatBoost,0.6474±0.0189,0.2522±0.0270,0.8520±0.0073,0.3082±0.0341,0.2136±0.0250
3,Bank,90%,CatBoost,0.5264±0.0179,0.0150±0.0100,0.8826±0.0064,0.4024±0.2211,0.0076±0.0051
4,Blood,0%,CatBoost,0.7455±0.0972,0.4545±0.1564,0.7816±0.0680,0.5601±0.1995,0.3882±0.1598


,Dataset,Missing_Ratio,Model,roc_auc,f1,accuracy,precision,recall
0,Bank,0%,LightGBM,0.9238±0.0070,0.5423±0.0272,0.9069±0.0058,0.6372±0.0332,0.4723±0.0303
1,Bank,20%,LightGBM,0.7904±0.0156,0.4098±0.0281,0.8802±0.0065,0.4834±0.0343,0.3559±0.0288
2,Bank,50%,LightGBM,0.6428±0.0195,0.2579±0.0287,0.8706±0.0068,0.3917±0.0415,0.1924±0.0240
3,Bank,90%,LightGBM,0.5073±0.0194,0.0347±0.0156,0.8821±0.0065,0.4064±0.1449,0.0181±0.0083
4,Blood,0%,LightGBM,0.7017±0.0964,0.4426±0.1441,0.7543±0.0697,0.4817±0.1746,0.4155±0.1554


In [ ]:
from google.colab import runtime
runtime.unassign()